# 🏗️ Détection de Bâtiments — Entraînement U-Net

**Dataset :** WHU Building Dataset | **Architecture :** U-Net | **Colab + Google Drive**

### Structure dans votre Google Drive :
```
Mon Drive/
└── Data/
    ├── Train/ ├── Image/  └── Mask/
    ├── Val/   ├── Image/  └── Mask/
    └── Test/  ├── Image/  └── Mask/
```
Le dossier `output/` sera créé automatiquement dans `Mon Drive/`.

## Étape 1 — Vérification GPU + Installation des dépendances

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositif : {device}')
if device.type == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Aucun GPU — Allez dans : Exécution → Modifier le type d\'exécution → T4 GPU')

!pip install -q albumentations tqdm
print('\n Dépendances OK')

## Étape 2 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print(' Google Drive monté')

## Étape 3 — Configuration des chemins et hyperparamètres

In [ ]:
import os
from pathlib import Path

# Chemins Google Drive
DATA_DIR   = Path('/content/drive/MyDrive/Data')
OUTPUT_DIR = Path('/content/drive/MyDrive/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_IMG  = DATA_DIR / 'Train' / 'Image'
TRAIN_MASK = DATA_DIR / 'Train' / 'Mask'
VAL_IMG    = DATA_DIR / 'Val'   / 'Image'
VAL_MASK   = DATA_DIR / 'Val'   / 'Mask'

# Hyperparamètres
IMG_SIZE   = 512
BATCH_SIZE = 4
EPOCHS     = 50
LR         = 1e-4
PATIENCE   = 10

for folder in [TRAIN_IMG, TRAIN_MASK, VAL_IMG, VAL_MASK]:
    n = len(list(folder.iterdir()))
    print(f'  {str(folder).replace("/content/drive/MyDrive/", "")} → {n} fichiers')
print(f'\n  Output → {OUTPUT_DIR}')

## Étape 4 — Dataset et Augmentations

In [ ]:
import numpy as np
from PIL import Image as PILImage
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

class BuildingDataset(Dataset):
    """Charge les paires (image satellite, masque binaire) du dataset WHU."""
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.mask_dir  = Path(mask_dir)
        self.transform = transform
        exts = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
        self.images = sorted([f for f in self.image_dir.iterdir() if f.suffix.lower() in exts])
        assert len(self.images) > 0, f'Aucune image dans {image_dir}'

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img_path  = self.images[idx]
        mask_path = self.mask_dir / img_path.name
        image = np.array(PILImage.open(img_path).convert('RGB'))
        mask  = np.array(PILImage.open(mask_path).convert('L'))
        mask  = (mask > 127).astype(np.float32)
        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug['image']
            mask  = aug['mask'].unsqueeze(0)
        else:
            image = torch.from_numpy(image).permute(2,0,1).float() / 255.0
            mask  = torch.from_numpy(mask).unsqueeze(0)
        return image, mask

def get_transforms(img_size=512, train=True):
    if train:
        return A.Compose([
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.5, 1.0)),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])

train_ds = BuildingDataset(TRAIN_IMG, TRAIN_MASK, transform=get_transforms(IMG_SIZE, train=True))
val_ds   = BuildingDataset(VAL_IMG,   VAL_MASK,   transform=get_transforms(IMG_SIZE, train=False))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f' Train : {len(train_ds)} images  ({len(train_loader)} batches)')
print(f' Val   : {len(val_ds)} images  ({len(val_loader)} batches)')

## Étape 5 — Architecture U-Net

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.downs = nn.ModuleList(); self.ups = nn.ModuleList(); self.pool = nn.MaxPool2d(2, 2)
        ch = in_channels
        for f in features: self.downs.append(DoubleConv(ch, f)); ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f*2, f))
        self.final_conv = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for down in self.downs: x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x); skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x); skip = skips[i//2]
            if x.shape != skip.shape: x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([skip, x], dim=1); x = self.ups[i+1](x)
        return self.final_conv(x)

model = UNet().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f' U-Net créé — {n_params/1e6:.2f}M paramètres')

## Étape 6 — Loss, métriques, optimiseur

In [ ]:
import torch.optim as optim

class DiceBCELoss(nn.Module):
    """BCE + Dice pour segmentation binaire déséquilibrée."""
    def __init__(self, bce_weight=0.5):
        super().__init__(); self.bce_weight = bce_weight; self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        bce = self.bce(pred, target); p = torch.sigmoid(pred)
        inter = (p * target).sum(dim=(1,2,3)); union = p.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
        dice = 1 - (2*inter + 1e-6) / (union + 1e-6)
        return self.bce_weight * bce + (1 - self.bce_weight) * dice.mean()

def dice_score(pred, target, thr=0.5, eps=1e-6):
    pred = (torch.sigmoid(pred) > thr).float()
    inter = (pred * target).sum(dim=(1,2,3)); union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return ((2*inter + eps) / (union + eps)).mean()

def iou_score(pred, target, thr=0.5, eps=1e-6):
    pred = (torch.sigmoid(pred) > thr).float()
    inter = (pred * target).sum(dim=(1,2,3)); union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    return ((inter + eps) / (union + eps)).mean()

criterion = DiceBCELoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
print(' Loss : BCE + Dice  |  Optim : AdamW  |  Scheduler : CosineAnnealing')

## Étape 7 — Boucle d'entraînement (avec sauvegarde automatique sur Drive)

In [ ]:
from tqdm.notebook import tqdm

def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train(); total_loss = 0.0
    for images, masks in tqdm(loader, desc='Train', leave=False):
        images, masks = images.to(device), masks.to(device); optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            loss = criterion(model(images), masks)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval(); tot_loss = tot_dice = tot_iou = 0.0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device); preds = model(images)
        tot_loss += criterion(preds, masks).item()
        tot_dice += dice_score(preds, masks).item()
        tot_iou  += iou_score(preds, masks).item()
    n = len(loader); return tot_loss/n, tot_dice/n, tot_iou/n

history = {'train_loss':[], 'val_loss':[], 'val_dice':[], 'val_iou':[]}
best_dice = 0.0; no_improve = 0
CKPT_PATH = str(OUTPUT_DIR / 'best_model.pth')
print(f' Démarrage — {EPOCHS} époques max | Early stopping après {PATIENCE} sans amélioration\n')

for epoch in range(1, EPOCHS + 1):
    t_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
    v_loss, v_dice, v_iou = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    for k, v in zip(['train_loss','val_loss','val_dice','val_iou'], [t_loss, v_loss, v_dice, v_iou]):
        history[k].append(v)
    tag = ''
    if v_dice > best_dice:
        best_dice = v_dice; no_improve = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_dice': v_dice, 'val_iou': v_iou}, CKPT_PATH)
        tag = '   ✅ Sauvegardé'
    else:
        no_improve += 1
    print(f'Époque {epoch:>3}/{EPOCHS}  |  Loss train={t_loss:.4f}  val={v_loss:.4f}  |  Dice={v_dice:.4f}  IoU={v_iou:.4f}{tag}')
    if no_improve >= PATIENCE: print(f'\n  Early stopping à l\'époque {epoch}.'); break

print(f'\n Entraînement terminé — Meilleur Dice = {best_dice:.4f}')
print(f'   Modèle → {CKPT_PATH}')

## Étape 8 — Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Courbes d\'apprentissage', fontsize=14, fontweight='bold')
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss (BCE + Dice)'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history['val_dice'], color='green'); axes[1].set_title('Dice Score (Val)'); axes[1].grid(True)
axes[2].plot(history['val_iou'],  color='orange'); axes[2].set_title('IoU Score (Val)');  axes[2].grid(True)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show(); print(' Courbes sauvegardées dans Drive')

## Étape 9 — Aperçu des prédictions sur la validation

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
images, masks = next(iter(val_loader))
with torch.no_grad(): preds = torch.sigmoid(model(images.to(device))).cpu()
mean_n = np.array([0.485,0.456,0.406]); std_n = np.array([0.229,0.224,0.225])
n_show = min(4, BATCH_SIZE)
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4*n_show))
if n_show == 1: axes = axes[None, :]
for i in range(n_show):
    img  = (images[i].permute(1,2,0).numpy() * std_n + mean_n).clip(0,1)
    axes[i,0].imshow(img);                   axes[i,0].set_title('Image satellite'); axes[i,0].axis('off')
    axes[i,1].imshow(masks[i,0], cmap='gray'); axes[i,1].set_title('Masque réel');    axes[i,1].axis('off')
    axes[i,2].imshow((preds[i,0]>0.5).float(), cmap='gray'); axes[i,2].set_title('Prédiction U-Net'); axes[i,2].axis('off')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'validation_samples.png'), dpi=150, bbox_inches='tight')
plt.show()